# Vectorless Reasoning-Based RAG — a from-scratch walkthrough

This notebook builds a small, self-contained version of **vectorless RAG**: a retrieval-augmented generation pipeline that uses an LLM to *reason* over the natural structure of a document instead of searching a vector database of embeddings.

## Why bother?

In classic RAG you chunk your documents, embed every chunk into a high-dimensional vector, store those vectors in a database (pgvector, Pinecone, Chroma, etc.), and at query time you retrieve the top-k chunks by cosine similarity. It works, but it has well-known weaknesses:

- **Arbitrary chunk boundaries** can cut a sentence — or a table — in half.
- **Cosine similarity is opaque.** Two chunks scored 0.81 and 0.79; you can't really tell *why* one beat the other.
- **You need an embedding model, a vector DB, and a chunking strategy** — three moving parts that all have to stay in sync.

Vectorless RAG (popularized by [PageIndex](https://github.com/VectifyAI/PageIndex) and Microsoft's [writeup](https://techcommunity.microsoft.com/blog/azuredevcommunityblog/vectorless-reasoning-based-rag-a-new-approach-to-retrieval-augmented-generation/4502238)) drops all three:

| Classic vector RAG | Vectorless reasoning RAG |
|---|---|
| Chunk → embed → store in vector DB | Keep natural pages/sections |
| Cosine similarity to retrieve top-k | LLM reasons over a tree-of-contents to pick pages |
| Opaque "why was this retrieved?" | Fully traceable: the LLM names the pages it chose and why |
| Needs embedding model + vector store | Just an LLM and a PDF |

The tradeoff: **more LLM calls per query** (you're paying for reasoning instead of a pgvector lookup). It's a great fit for small/medium corpora where explainability and accuracy matter more than millisecond latency — finance, legal, compliance, research.

## What we'll build

A four-step pipeline:

1. **Load** a PDF and split it by *page* (no chunking, no embeddings).
2. **Index** — ask a cheap LLM to write one short summary per page; this becomes our "table of contents."
3. **Retrieve** — give the whole ToC to a stronger LLM and let it *reason* about which pages to read.
4. **Answer** by reading only those pages, with page-number citations.

Sample document: **[NIST AI 600-1 — Artificial Intelligence Risk Management Framework: Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)** (U.S. NIST, public domain). It's on-topic for GenAI, heavily structured (numbered sections, suggested actions, page-aligned content), and safe to redistribute. Already included at `data/nist_ai_600-1.pdf`.


## 0. Setup

You need an Anthropic API key. There are three ways to provide it, in order of preference:

1. **Environment variable** before launching Jupyter:
   ```bash
   export ANTHROPIC_API_KEY=sk-ant-...
   ```
2. **A `.env` file** in this folder — `python-dotenv` will load it automatically.
3. **Interactive prompt** — if the cell below doesn't find a key, it'll ask for one via `getpass`, which hides what you type and (crucially) doesn't write it into the notebook file.

### What the cell below does, line by line

- `import os, json, pathlib, getpass` — standard library helpers for env vars, JSON, paths, and the hidden-input prompt.
- `from dotenv import load_dotenv; load_dotenv()` — looks for a `.env` file in the current folder and loads any `KEY=VALUE` pairs into environment variables. Silently does nothing if no `.env` exists.
- `import anthropic` and `from pypdf import PdfReader` — the two external libraries we'll actually use: the official Claude SDK and a pure-Python PDF text extractor.
- The four `PATH`/`MODEL` constants are gathered in one place so you can swap them easily. **`INDEX_MODEL`** (Haiku, cheap) runs *once per page* at indexing time. **`ANSWER_MODEL`** (Sonnet, smarter) runs *once per query* for both retrieval reasoning and the final cited answer. This split is the cost/quality trick that makes the pipeline practical.
- The `if not os.environ.get("ANTHROPIC_API_KEY")` block is the fail-fast guard. Without it, `anthropic.Anthropic()` would silently construct a client with no credentials, and you'd get a confusing `TypeError` four cells later when the first real API call happens.
- `client = anthropic.Anthropic()` — creates the API client; it reads the key from the env var we just verified.


In [1]:
import os, json, pathlib, getpass
from dotenv import load_dotenv
load_dotenv()  # pulls ANTHROPIC_API_KEY from a .env file if present

import anthropic
from pypdf import PdfReader

PDF_PATH     = "data/nist_ai_600-1.pdf"
TOC_PATH     = "data/toc.json"         # cache so we only pay for indexing once
INDEX_MODEL  = "claude-haiku-4-5"      # cheap & fast — used once per page at index time
ANSWER_MODEL = "claude-sonnet-4-6"     # smarter — used for retrieval reasoning + final answer

# Fail fast if the key is missing. anthropic.Anthropic() will silently succeed
# without a key and only blow up later at the first real API call.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your ANTHROPIC_API_KEY: ").strip()

assert os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-"), \
    "ANTHROPIC_API_KEY is missing or malformed."

client = anthropic.Anthropic()
print("Anthropic SDK:", anthropic.__version__, "— ready.")


Anthropic SDK: 0.104.1 — ready.


## 1. Load the PDF, page by page

In classic RAG this is where you'd start splitting the document into ~500-token chunks with some overlap. We're not going to do that.

**The page is our atomic unit of retrieval.** It's a natural boundary the document's author chose: a page usually ends at a section break or a paragraph break, tables sit on one page, footnotes stay with their text. We extract the text from each page with `pypdf` and keep a `(page_number, text)` pair. That's it — no embeddings, no chunking, no overlap.

### What the cell below does

- `load_pages(pdf_path)` opens the PDF with `pypdf` and walks every page with `enumerate(..., start=1)` so our page numbers match what a human would see on the printed document (1-indexed).
- For each page we call `page.extract_text()` and `.strip()` it. The `or ""` handles weird pages where `extract_text()` returns `None` (which happens occasionally with scanned/image-only pages).
- The `if text:` filter drops any pages that came out completely empty — those would only confuse the indexing step.
- We return a list of plain dicts `{"page": N, "text": "..."}`. Boring, deliberately. The whole point of vectorless RAG is that the data structure stays simple.
- After defining the function, we run it on the NIST PDF and print a quick sample from page 11 so you can see what the extracted text actually looks like (it's a bit ragged — PDF text extraction always is — but plenty good enough for the LLM to read).


In [2]:
def load_pages(pdf_path: str) -> list[dict]:
    """Return a list of {'page': N, 'text': ...} dicts, one per non-empty page."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(f"\n--- Sample of what page {pages[10]['page']} looks like ---")
print(pages[10]["text"][:500], "...")


Loaded 64 non-empty pages

--- Sample of what page 11 looks like ---
7 
unethical behavior. Text-to-image models also make it easy to create images that could be used to 
promote dangerous or violent messages. Similar concerns are present for other GAI media, including 
video and audio. GAI may also produce content that recommends self-harm or criminal/illegal activities.  
Many current systems restrict model outputs to limit certain content or in response to certain prompts, 
but this approach may still produce harmful recommendations in response to other less-e ...


## 2. Build a tree-of-contents (one summary per page)

This is the **indexing** step. For each page we ask a cheap, fast model (Haiku) to produce a structured summary:

- `title` — a short label for the page (≤ 10 words)
- `summary` — 2-3 sentences describing what's *on* this specific page (topics, sections, entities)
- `keywords` — distinctive words/phrases someone might search for

Why this works: the model is *much* better at writing a faithful one-paragraph summary of a page than the embedding-based alternative is at picking the right chunk via cosine similarity. The summaries are concise enough that the **entire index for a 64-page document fits in a single prompt** — which is exactly what makes the retrieval step in the next section possible.

We cache the result to `data/toc.json`. Indexing costs roughly one Haiku call per page (~64 calls for this PDF), so you don't want to redo it every kernel restart. Delete the file to regenerate.

### What the cell below does

- **`SUMMARY_PROMPT`** is the instruction we send to the indexing model. It explicitly asks for JSON with three named fields, and tells the model "return ONLY the JSON, no preamble" — small models love to add a friendly "Sure! Here's your JSON:" otherwise. The page text is wrapped in `"""..."""` to mark its boundaries clearly.
- **`_extract_json(raw)`** is a small defensive helper. Even with "return only JSON" in the prompt, models occasionally wrap their reply in ```` ```json ... ``` ```` fences, or add a leading newline, or prepend a sentence. This function:
  1. Strips backtick fences if present and removes the leading `json` language tag.
  2. As a last-ditch fallback, slices between the first `{` and last `}` so we always get *something* parseable.
- **`summarize_page(page)`** is the actual API call. We pass `max_tokens=400` (summaries are short) and truncate the input text to 6000 chars to be safe on very dense pages. After the response comes back we run it through `_extract_json` and stamp the page number on the result so we don't have to track it separately.


In [3]:
SUMMARY_PROMPT = """You are indexing a single page of a document for later retrieval.

Read the page text below and return a JSON object with these fields:
- "title":   a short (<= 10 words) title describing what this page is about
- "summary": 2-3 sentences capturing the main content; mention specific topics, sections, or entities
- "keywords": a list of 5-10 distinctive keywords or phrases someone might search for to find this page

Return ONLY the JSON, no preamble.

PAGE TEXT:
\"\"\"
{text}
\"\"\"
"""

def _extract_json(raw: str) -> dict:
    """Robustly pull the first {...} JSON object out of an LLM response."""
    raw = raw.strip()
    # Strip ```json ... ``` fences if present
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    # Fallback: slice between the first { and the last }
    if not raw.startswith("{"):
        start, end = raw.find("{"), raw.rfind("}")
        if start != -1 and end != -1:
            raw = raw[start : end + 1]
    return json.loads(raw)

def summarize_page(page: dict) -> dict:
    """Call the indexing model to produce a structured summary for one page."""
    msg = client.messages.create(
        model=INDEX_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": SUMMARY_PROMPT.format(text=page["text"][:6000])}],
    )
    data = _extract_json(msg.content[0].text)
    data["page"] = page["page"]
    return data


### Now build the full table of contents

The cell above only *defined* the per-page indexing function — it didn't actually run anything. This next cell does the work.

### What the cell below does

- **`build_toc(pages, cache_path)`** is the orchestrator. It first checks if a cached `toc.json` exists; if so, it just loads it and returns immediately. That's the cost-saver — once you've indexed this PDF once, every subsequent kernel restart is free.
- If there's no cache, it loops over every page and calls `summarize_page()`. The `\r` carriage return in the progress print makes the line overwrite itself in a terminal (in Jupyter it just prints one line per page — not the prettiest but harmless).
- The `try/except` around `summarize_page` is there because indexing 64+ pages is the most likely place to hit a transient API error. If one page fails we substitute a placeholder entry instead of crashing the whole indexing run — much friendlier than starting over.
- After the loop, we write the whole list to `data/toc.json` so the next run is instant.
- Finally we print the first three entries so you can eyeball that the summaries look reasonable. If they look generic or wrong, your `SUMMARY_PROMPT` probably needs work — this is the moment to catch quality issues, not at query time.


In [4]:
def build_toc(pages: list[dict], cache_path: str) -> list[dict]:
    cache = pathlib.Path(cache_path)
    if cache.exists():
        print(f"Loading cached ToC from {cache_path}")
        return json.loads(cache.read_text())

    toc = []
    for p in pages:
        print(f"  indexing page {p['page']}/{len(pages)}...", end="\r")
        try:
            toc.append(summarize_page(p))
        except Exception as e:
            print(f"\n  page {p['page']} failed: {e}")
            toc.append({"page": p["page"], "title": "(indexing failed)", "summary": "", "keywords": []})
    cache.write_text(json.dumps(toc, indent=2))
    print(f"\nSaved ToC to {cache_path}")
    return toc

toc = build_toc(pages, TOC_PATH)
print(f"\nIndexed {len(toc)} pages. Here are the first three entries:\n")
for entry in toc[:3]:
    print(json.dumps(entry, indent=2))
    print("---")


Loading cached ToC from data/toc.json

Indexed 64 pages. Here are the first three entries:

{
  "title": "NIST AI Risk Management Framework for Generative AI",
  "summary": "This NIST publication (AI 600-1) presents a framework for managing risks associated with generative artificial intelligence systems. It is part of NIST's broader initiative on trustworthy and responsible AI, providing guidance on responsible AI development and deployment.",
  "keywords": [
    "NIST AI 600-1",
    "generative artificial intelligence",
    "AI risk management",
    "trustworthy AI",
    "responsible AI",
    "AI framework",
    "generative AI profile",
    "AI governance"
  ],
  "page": 1
}
---
{
  "title": "NIST AI Risk Management Framework for Generative AI",
  "summary": "This NIST publication (AI 600-1) from July 2024 presents a specialized risk management framework tailored for generative artificial intelligence systems. Published by the U.S. Department of Commerce's National Institute of Stand

## 3. Reasoning-based retrieval

This is the part that replaces your vector database.

We take the user's question, paste the **entire table of contents** into a prompt, and ask a stronger model (Sonnet) to pick the page numbers it thinks are most relevant — *and to explain its reasoning out loud*.

A few things to notice about this design:

- **No similarity scores, no top-k threshold.** The model picks however many pages it thinks the question needs (usually 2-8). For a narrow factual question it might pick one page; for a broad "summarize all governance recommendations" question it might pick a dozen.
- **The reasoning is visible.** You get a paragraph explaining *why* those pages were chosen, which means you can audit retrieval failures the same way you'd review a junior analyst's work — instead of squinting at embedding distances.
- **Latency and cost scale with ToC size, not corpus full-text size.** As long as your summaries fit in one prompt (~comfortably 200+ pages with Sonnet's context window), retrieval is a single LLM call.

For really big corpora the trick is to make the ToC hierarchical (summarize sections, then pages within sections) — but for a 64-page document, flat is fine.

### What the cell below does

- **`RETRIEVAL_PROMPT`** asks the model to return a JSON object with two fields: `reasoning` (free-text paragraph) and `pages` (list of page numbers). Asking for the reasoning *first* nudges the model to actually think before naming pages — this is a chain-of-thought trick that measurably improves retrieval quality.
- **`retrieve_pages(question, toc, ...)`** compacts each ToC entry into a single line `- p.N: <title> — <summary>` so the whole index stays under the context limit even for hundreds of pages. It then sends the question + compacted ToC to the answer model and parses the JSON reply with the same `_extract_json` helper we defined earlier.
- We use `ANSWER_MODEL` (Sonnet) here, not `INDEX_MODEL` (Haiku) — retrieval reasoning is the highest-leverage step in the whole pipeline, so it's worth spending on a smarter model.


In [5]:
RETRIEVAL_PROMPT = """You are a retrieval agent for a document. Below is a table of contents where each entry corresponds to a single page.

Your job: choose the pages most likely to contain information needed to answer the user's question. Reason about which sections are relevant.

Return a JSON object with:
- "reasoning": a short paragraph explaining which sections of the document are relevant and why
- "pages": a list of page numbers to read (typically 1-5 pages; only include pages that genuinely help)

Return ONLY the JSON.

USER QUESTION:
{question}

TABLE OF CONTENTS:
{toc}
"""

def retrieve_pages(question: str, toc: list[dict], answer_model: str = ANSWER_MODEL) -> dict:
    """Ask the model which pages to read, and why."""
    # Compact ToC representation: one line per page
    toc_text = "\n".join(
        f"- p.{e['page']}: {e['title']} — {e['summary']}" for e in toc
    )
    msg = client.messages.create(
        model=answer_model,
        max_tokens=600,
        messages=[{"role": "user", "content": RETRIEVAL_PROMPT.format(question=question, toc=toc_text)}],
    )
    return _extract_json(msg.content[0].text)


## 4. Answer synthesis

Once we know which pages to read, we pull their *full text* (not the summaries — the actual content) and hand it to the model along with the original question.

Two key constraints in the answer prompt:

1. **"Use ONLY the document pages provided."** This is what makes the output grounded — the model shouldn't fall back on what it remembers about NIST from training.
2. **"Cite the page numbers you used, like (p. 12)."** This is the explainability payoff. Every claim in the answer is traceable to a specific page you can open and read yourself.

### What the cell below does

- **`ANSWER_PROMPT`** sets up the role: answer using only the provided pages, admit ignorance if they're insufficient, cite inline.
- **`ask(question, toc, pages, verbose=True)`** is the end-to-end helper that ties steps 3 and 4 together.
  1. It calls `retrieve_pages()` to get the list of relevant page numbers + the model's reasoning.
  2. It builds a `page_lookup` dict (`{page_num: full_text}`) so it can grab the actual content of the chosen pages without scanning the list every time.
  3. It assembles a `pages_text` block formatted with `=== Page N ===` separators — these explicit boundaries help the model attribute facts to specific pages when it writes citations.
  4. When `verbose=True` (the default), it prints the retrieval reasoning and the chosen pages *before* generating the answer, so you can see what the model decided to read.
  5. Finally it sends question + pages to `ANSWER_MODEL` and returns the cited answer.

The whole pipeline ends up being two LLM calls per query: one for retrieval, one for synthesis.


In [6]:
ANSWER_PROMPT = """You are answering a user's question using ONLY the document pages provided below. If the pages do not contain enough information, say so honestly.

Cite the page numbers you used inline, like (p. 12). Be concise and specific.

USER QUESTION:
{question}

PAGES:
{pages}
"""

def ask(question: str, toc: list[dict], pages: list[dict], verbose: bool = True) -> str:
    """End-to-end: retrieve relevant pages, then synthesize a cited answer."""
    selection = retrieve_pages(question, toc)
    chosen = selection["pages"]
    page_lookup = {p["page"]: p["text"] for p in pages}
    pages_text = "\n\n".join(
        f"=== Page {n} ===\n{page_lookup.get(n, '(page not found)')}" for n in chosen
    )

    if verbose:
        print("Retrieval reasoning:")
        print(" ", selection["reasoning"])
        print("Chosen pages:", chosen)
        print()

    msg = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": ANSWER_PROMPT.format(question=question, pages=pages_text)}],
    )
    return msg.content[0].text.strip()


## 5. Try it

Three example queries against the NIST GenAI Profile, picked to stress different shapes of question:

| # | Question type | What to watch for |
|---|---|---|
| 1 | **Broad / list-style** — "what governance actions?" | Retrieval should pull a *cluster* of related pages (the governance section). Answer will be long and organized into categories. |
| 2 | **Specific / enumerative** — "which risks are unique to GAI?" | Retrieval should hit the few pages where NIST defines its risk taxonomy. Answer should be a numbered list with one citation per item. |
| 3 | **Narrow definitional** — "define 'confabulation'" | Retrieval should pick mostly the one page that defines the term, plus a couple of supporting pages. Answer should be short and pinpoint-cited. |

For each query, the printed output has two parts:
1. **Retrieval reasoning** — the model thinking out loud about which pages to look at. *This is the part you can't get from a vector DB.* If the answer is later wrong, this is what you inspect to figure out whether retrieval failed or synthesis failed.
2. **The cited answer** — every factual claim tagged with `(p. N)` so you can open the source PDF and verify.


### Query 1 — broad governance question

Expect the retrieval step to pick a wide span of pages from the GOVERN section of the AI RMF (roughly pp. 16-26 in this PDF). The answer should organize the governance actions into themes (legal, risk assessment, transparency, third-party, etc.) and cite the page where each recommendation appears.


In [7]:
question = "What are some suggested actions for governing risks specific to generative AI?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The user is asking about suggested actions for governing risks specific to generative AI. This is directly addressed in the AI RMF governance sections (GOVERN functions) which contain specific suggested actions organized by subcategories. Pages 17-26 cover the GOVERN function subcategories (GV-1.1 through GV-6.2) with explicit suggested actions tables. Page 16 introduces the structure of suggested actions. Pages 18, 19, 22, 23, 24, and 25 are particularly relevant as they outline specific governance policies, oversight frameworks, and risk management actions for GAI. Page 51 and 52 in the appendix also discuss primary GAI governance considerations.
Chosen pages: [17, 18, 19, 22, 23, 24, 25]



Based on the provided pages, here are key suggested actions for governing risks specific to generative AI (GAI), organized by category:

**Legal & Policy Alignment**
- Align GAI development with applicable laws on data privacy, copyright, and intellectual property (p. 17)
- Establish transparent acceptable use policies addressing illegal applications (p. 19)
- Establish policies to prevent GAI systems from generating CSAM, NCII, or other unlawful content (p. 19)

**Risk Assessment & Thresholds**
- Define risk tiers for GAI considering factors like information integrity, psychological impacts, malicious use potential, and security vulnerabilities (p. 18)
- Establish minimum performance/assurance thresholds for deployment approval ("go/no-go" policies) (p. 18)
- Devise a plan to halt development or deployment of GAI systems posing unacceptable risk (p. 19)

**Oversight & Human-AI Configuration**
- Implement independent evaluations proportional to identified risks (p. 22)
- Engage in thre

### Query 2 — enumerative "what are the risks?" question

This question maps to a very specific part of the document: NIST's taxonomy of risks unique to or exacerbated by GAI. Expect retrieval to pick just the few pages where that list is defined (~pp. 4-9). The answer should come back as a numbered list — ideally close to a dozen items — with each one cited.


In [8]:
question = "Which risks does NIST identify as unique to or exacerbated by generative AI compared to traditional AI?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The user is asking specifically about risks that NIST identifies as unique to or exacerbated by generative AI compared to traditional AI. The most relevant pages are those that directly describe and categorize GAI-specific risks. Page 4 provides a comprehensive overview of risks unique to or exacerbated by GAI. Page 5 explicitly mentions 'risks novel to or exacerbated by GAI technologies.' Pages 8 and 9 outline the nine major risk categories for GAI systems. Pages 6 and 7 describe the dimensions and characteristics of GAI risks. Pages 10-15 provide detailed descriptions of specific risk categories like confabulation, harmful content, data privacy, environmental impacts, bias, human-AI configuration, information integrity, and cybersecurity.
Chosen pages: [4, 5, 8, 9, 7]



## NIST-Identified Risks Unique to or Exacerbated by Generative AI

NIST identifies **12 risks** unique to or exacerbated by GAI (p. 4–5):

1. **CBRN Information or Capabilities** – Eased access to chemical, biological, radiological, or nuclear weapons knowledge, potentially lowering barriers for malicious actors without formal scientific training (p. 4, 5).

2. **Confabulation** – Production of confidently stated but false content ("hallucinations"), misleading users (p. 4).

3. **Dangerous, Violent, or Hateful Content** – Eased production of violent, inciting, radicalizing, or threatening content, including self-harm recommendations (p. 4).

4. **Data Privacy** – Leakage or unauthorized disclosure of biometric, health, location, or other sensitive personal data (p. 4).

5. **Environmental Impacts** – High compute resource utilization in training/operating GAI models with adverse ecosystem effects (p. 4).

6. **Harmful Bias or Homogenization** – Amplification of historical and societa

### Query 3 — narrow definitional question

A pinpoint question like "define confabulation" should produce *narrow* retrieval — ideally just the one page where NIST defines the term, plus maybe one or two pages that reference it. This is where vectorless RAG really shines over top-k vector retrieval: the model can confidently pick one page instead of being forced to return some arbitrary number of "most similar" chunks.


In [9]:
question = "How does the document define 'confabulation' and why is it a risk?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The question asks specifically about how 'confabulation' is defined and why it is a risk. Page 10 is explicitly titled 'Confabulation in Generative AI Systems' and directly defines the term and discusses its risks. Page 8 also lists confabulation/hallucinations as one of the nine major risk categories, which may provide a summary-level definition and risk context. These two pages are the most directly relevant. Page 39 also mentions confabulation as a risk in the context of model explanation and validation measures, which may add some context.
Chosen pages: [10, 8, 39]



## Definition of Confabulation

The document defines **confabulation** as "a phenomenon in which GAI systems generate and confidently present erroneous or false content in response to prompts," including outputs that diverge from input prompts or contradict previously generated statements in the same context (p. 10). It is colloquially known as "hallucinations" or "fabrications" (p. 8).

## Why It Occurs

Confabulation is a **natural byproduct of how generative models work**: they generate outputs that approximate the statistical distribution of their training data (e.g., LLMs predict the next token/word), which can produce both accurate and inaccurate results. It is especially problematic with open-ended prompts and domains requiring specialized expertise (p. 10).

## Why It Is a Risk

The document highlights several reasons confabulation is dangerous:

- **Users may act on false information** due to the confident tone of GAI responses (p. 10)
- **High-stakes domains** like healthcare

## Where to go next

Now that the core loop works, the natural extensions are:

- **Hierarchical ToC.** For documents longer than a few hundred pages, summarize *sections* (groups of pages) on top of page summaries. Retrieval then walks the tree top-down — first pick the section, then the pages within it — instead of scanning every page summary in one call.
- **Multi-document corpora.** Add a per-document description and a routing step that picks the right document before picking pages. PageIndex calls this the "library" layer.
- **Prompt caching.** The ToC is reused across every query — wrap it in an Anthropic [`cache_control`](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching) block to drop retrieval cost by ~90%.
- **Compare against vector RAG.** Run both pipelines on the same questions and look at: answer quality, latency, cost per query, and (the most useful metric) how often you can trace *why* a given chunk was retrieved. That's where vectorless usually wins.
- **Swap in your own document.** Change `PDF_PATH`, delete `data/toc.json`, and re-run. Anything with clear section structure (financial filings, legal contracts, technical specs, research papers) is a good candidate.
